# Task A — MediaPipe tracker quality audit (Kaggle)

Runs the tracker-quality audit against the Stage 1 v3 results.

**Inputs the notebook needs**:
1. `stage1v3_results.json` — per-signer CER source.  Either present at one of the standard paths, or attached as a Kaggle dataset.
2. `skeleton_features_t32.pt` — the MediaPipe landmark cache.  If absent, Cell 4 rebuilds it from scratch (~30 min).  The rebuilt cache is also what Stage 4 fusion needs later.

Two run modes:
- **cache mode (default, ~10 s)** — uses the skeleton cache.  Adequate for the Pearson(visibility, CER) verdict.
- **reextract mode (~30 min CPU)** — re-runs MediaPipe natively + renders 5 overlay GIFs per audited signer.

**Kaggle settings**: leave GPU **off** for the audit itself.  Turn GPU **on** only if you're rebuilding the skeleton cache from scratch (MediaPipe is CPU but the cache build benefits from a faster compute instance).

## Cell 1 — Install + clone

In [ ]:
%%capture
!pip install editdistance huggingface_hub hgtk 'mediapipe>=0.10.0' scipy imageio --quiet

import sys, os, glob
!rm -rf /kaggle/working/wita_v2
!git clone -b iterative-ablation "https://github.com/Gaurs86/WiTA-v2.git" '/kaggle/working/wita_v2'
sys.path.insert(0, '/kaggle/working')

## Cell 2 — Upload your logs.zip OR attach as a Kaggle dataset

If your Stage 1 v3 kernel was committed: **Data → Add Data → Notebook output** and select that kernel.  Skip to Cell 3.

If you only have the local `logs.zip` (because the session expired): upload it as a private Kaggle dataset:
1. *Data → Add Data → Upload* → drop `logs.zip` → name it (e.g. `wita-stage1v3-logs`).
2. Kaggle mounts it at `/kaggle/input/<dataset-name>/`.
3. The cell below extracts the zip into `/kaggle/working/logs/` if needed.

In [ ]:
import os, glob, zipfile, shutil

def _first(*candidates):
    for c in candidates:
        if c and os.path.exists(c):
            return c
    return None

# Look for stage1v3_results.json in the obvious places first.
RESULTS_PATH = _first(
    '/kaggle/working/logs/stage1v3_results.json',
    '/kaggle/working/stage1v3_results.json',
    *glob.glob('/kaggle/input/*/logs/stage1v3_results.json'),
    *glob.glob('/kaggle/input/*/stage1v3_results.json'),
    *glob.glob('/kaggle/input/*/kaggle/working/logs/stage1v3_results.json'),
)

# If still missing, look for a logs.zip in /kaggle/input/* and extract it.
if RESULTS_PATH is None:
    candidates = glob.glob('/kaggle/input/*/logs.zip') + glob.glob('/kaggle/input/*/*/logs.zip')
    if candidates:
        zip_path = candidates[0]
        target = '/kaggle/working/logs_extracted'
        os.makedirs(target, exist_ok=True)
        with zipfile.ZipFile(zip_path, 'r') as z:
            z.extractall(target)
        # Find the extracted results.json.
        RESULTS_PATH = _first(
            *glob.glob(f'{target}/**/stage1v3_results.json', recursive=True),
        )
        print(f'extracted {zip_path} -> {target}')

assert RESULTS_PATH, (
    'stage1v3_results.json not found.  Either attach the Stage 1 v3 kernel\n'
    'output as a notebook-output dataset, or upload logs.zip as a private\n'
    'Kaggle dataset (see Cell 2 instructions above).'
)
print(f'results path : {RESULTS_PATH}')

# Quick sanity check on the JSON: should have 15 entries with best_per_signer_val_cer.
import json
with open(RESULTS_PATH) as f:
    _r = json.load(f)
n_per_signer_entries = sum(1 for e in _r if e.get('best_per_signer_val_cer'))
print(f'entries      : {len(_r)}  with per-signer CER : {n_per_signer_entries}')
no_dann_signers = set()
for e in _r:
    if e.get('variant') == 'no_dann':
        no_dann_signers.update((e.get('best_per_signer_val_cer') or {}).keys())
print(f'no_dann covers {len(no_dann_signers)} signers (expected 39)')

## Cell 3 — Configure paths

Edit `CACHE_PATH` if your skeleton cache lives somewhere else.  Leave the default if you don't have it — Cell 4 will build it.

In [ ]:
CACHE_PATH = _first(
    '/kaggle/working/skeleton_features_t32.pt',
    *glob.glob('/kaggle/input/*/skeleton_features_t32.pt'),
    *glob.glob('/kaggle/input/*/kaggle/working/skeleton_features_t32.pt'),
) or '/kaggle/working/skeleton_features_t32.pt'

OUTPUT_DIR = '/kaggle/working/tracker_audit'
MANIFEST   = '/kaggle/working/audit_clips.json'
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f'cache path   : {CACHE_PATH}  (exists={os.path.exists(CACHE_PATH)})')
print(f'output dir   : {OUTPUT_DIR}')

## Cell 4 — (If cache missing) build the skeleton cache from scratch

Skipped automatically if `CACHE_PATH` already points at a real file.

Wall-clock: ~30 min on Kaggle (MediaPipe is CPU, but downloading 39 × 2 zips is the real bottleneck).  No GPU required.  The cache that lands at `/kaggle/working/skeleton_features_t32.pt` is the same one Stage 4 fusion will consume — not wasted compute.

In [ ]:
import torch, random, numpy as np
from wita_v2.configs.default import Config, DataConfig, EncoderConfig, TrainConfig

if not os.path.exists(CACHE_PATH):
    SEED = 42
    random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

    cfg = Config(
        data=DataConfig(
            hf_repo_id='yewon816/WiTA', lang='english',
            max_zips=None, max_frames=64,
            train_split=0.90, seed=SEED,
        ),
        encoder=EncoderConfig(arch='siglip'),
        train=TrainConfig(num_epochs=1, batch_size=1, seed=SEED,
                          checkpoint_dir='/tmp/_unused'),
    ).build()

    from wita_v2.datasets.subject_splits import stream_and_index_with_subjects
    from wita_v2.datasets.skeleton_cache  import extract_skeleton_features

    samples = stream_and_index_with_subjects(cfg)
    print(f'Streamed {len(samples)} clips across {len({s for _,_,s in samples})} subjects')

    extract_skeleton_features(
        samples=samples, out_path=CACHE_PATH,
        T_native=32, dtype=torch.float16,
    )
else:
    print(f'cache already at {CACHE_PATH} — skipping rebuild')

## Cell 5 — Run the audit (cache mode, ~10 s)

Uses the cache from Cell 4 plus per-signer CER from Cell 2's `stage1v3_results.json`.  Emits `per_signer_quality.csv`, `quality_vs_cer.png`, and `audit_summary.json` in `/kaggle/working/tracker_audit/`.

In [ ]:
!python /kaggle/working/wita_v2/scripts/audit_tracker_quality.py \
    --stage1v3-results {RESULTS_PATH} \
    --cache-path       {CACHE_PATH} \
    --output-dir       {OUTPUT_DIR} \
    --audit-manifest   {MANIFEST} \
    --mode             cache

## Cell 6 — Inspect the result inline

In [ ]:
import os, pandas as pd
from IPython.display import Image, display

df = pd.read_csv(os.path.join(OUTPUT_DIR, 'per_signer_quality.csv'))
df = df.sort_values('val_cer', ascending=False)
display(df)
display(Image(filename=os.path.join(OUTPUT_DIR, 'quality_vs_cer.png')))

## Cell 7 — (Optional) reextract mode for native-resolution metrics + overlay GIFs

Only run if Cell 5's verdict was `tracker_partial` (Pearson |r| between 0.3 and 0.5).  The GIFs are useful for eyeballing whether the hard-signer trajectories look genuinely ambiguous or merely drop-out-corrupted.

Wall-clock: ~30 min on Kaggle CPU.  Network: re-downloads the WiTA zips for the 16 audited signers (Kaggle's `huggingface_hub` cache handles this transparently).

In [ ]:
# Uncomment to run.
# !python /kaggle/working/wita_v2/scripts/audit_tracker_quality.py \
#     --stage1v3-results {RESULTS_PATH} \
#     --output-dir       {OUTPUT_DIR} \
#     --audit-manifest   {MANIFEST} \
#     --max-overlays     5 \
#     --mode             reextract

## Cell 8 — Verdict and next step

Read the printed verdict from Cell 5:
- **tracker_dominant** — keep `visibility_gate: true` in `configs/stage3.yaml` (current default).
- **tracker_null**     — flip `VISIBILITY_GATE = False` in Cell 2 of `run_stage3_cv.ipynb`; cache rebuilds with 1152-dim input.
- **tracker_partial**  — inspect the overlay GIFs (Cell 7) and decide by eye.

Commit this kernel so the audit's CSV, PNG, and `audit_summary.json` are downloadable for the dissertation appendix.
Also commit so the rebuilt `skeleton_features_t32.pt` becomes attachable to your Stage 3 / Stage 4 kernels later — the file is identical to what Stage 1 v3 produced, just from a fresh run.